In [ ]:
# 01 – YAML to CSV Conversion

Purpose:
- Parse Cricsheet IPL YAML files
- Extract match-level and ball-by-ball data
- Convert nested YAML into flat tabular structure

Inputs:
- data/raw/yaml/*.yaml

Outputs:
- data/processed/matches_raw.csv
- data/processed/deliveries_raw.csv



In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from config import *
import yaml
import pandas as pd

In [3]:
matches = []
deliveries = []

match_id = 1
file_count = 0

In [4]:
for file in YAML_DIR.iterdir():
    if not file.is_file():
        continue

    file_count += 1
    if file_count % 50 == 0:
        print(f"Processed {file_count} files")

    try:
        with open(file, "r", encoding="utf-8") as f:
            data = yaml.safe_load(f)
    except Exception:
        continue

    if not isinstance(data, dict):
        continue
    if "info" not in data or "innings" not in data:
        continue

    info = data["info"]

    match_date = info["dates"][0]
    season = int(match_date[:4]) if isinstance(match_date, str) else match_date.year

    matches.append({
        "match_id": match_id,
        "season": season,
        "city": info.get("city"),
        "venue": info.get("venue"),
        "date": match_date,
        "team1": info["teams"][0],
        "team2": info["teams"][1],
        "toss_winner": info["toss"]["winner"],
        "toss_decision": info["toss"]["decision"],
        "winner": info.get("outcome", {}).get("winner")
    })

    for innings in data["innings"]:
        innings_name = list(innings.keys())[0]
        innings_data = innings[innings_name]
        batting_team = innings_data["team"]

        for delivery in innings_data["deliveries"]:
            ball_no = list(delivery.keys())[0]
            ball = delivery[ball_no]

            deliveries.append({
                "match_id": match_id,
                "innings": innings_name,
                "batting_team": batting_team,
                "over": int(ball_no),
                "ball": int(round((ball_no % 1) * 10)),
                "batsman": ball.get("batsman"),
                "bowler": ball.get("bowler"),
                "non_striker": ball.get("non_striker"),
                "runs_batsman": ball["runs"]["batsman"],
                "runs_extras": ball["runs"]["extras"],
                "runs_total": ball["runs"]["total"],
                "extras_type": (
                    list(ball.get("extras", {}).keys())[0]
                    if "extras" in ball else None
                ),
                "player_dismissed": ball.get("wicket", {}).get("player_out"),
                "dismissal_kind": ball.get("wicket", {}).get("kind")
            })

    match_id += 1

Processed 50 files
Processed 100 files
Processed 150 files
Processed 200 files
Processed 250 files
Processed 300 files
Processed 350 files
Processed 400 files
Processed 450 files
Processed 500 files
Processed 550 files
Processed 600 files
Processed 650 files
Processed 700 files
Processed 750 files
Processed 800 files
Processed 850 files
Processed 900 files
Processed 950 files
Processed 1000 files
Processed 1050 files
Processed 1100 files
Processed 1150 files


In [5]:
matches_df = pd.DataFrame(matches)
deliveries_df = pd.DataFrame(deliveries)

print("Matches shape:", matches_df.shape)
print("Deliveries shape:", deliveries_df.shape)

Matches shape: (1169, 10)
Deliveries shape: (278205, 14)


In [6]:
matches_df.to_csv(PROCESSED_DIR / "matches_raw.csv", index=False)
deliveries_df.to_csv(PROCESSED_DIR / "deliveries_raw.csv", index=False)

print("Raw CSV files created successfully")

Raw CSV files created successfully
